# Démo GPS générique sans phase

Ce notebook montre le chemin le plus simple pour utiliser `xyt_gps` sur des traces GPS ordinaires, sans expérimentation Déclic, sans découpage en phases et sans configuration de projet complexe.

L'objectif est de comprendre le flux minimal : charger des données test, valider leur structure, transformer les traces en tables de mobilité, contrôler la qualité, puis produire les premiers indicateurs.

Le jeu de données utilisé est le testset fourni du dépôt. Il est pseudonymisé par `load_sample_gps()` et sert à apprendre le package sans disposer d’un export GPS complet.


## 1. Préparer l'environnement

Cette cellule permet de lancer le notebook depuis la racine du dépôt ou depuis `package-xyt-gps/examples`. Elle force l'utilisation du code local dans `src/`.


In [ ]:
from pathlib import Path
import sys


def find_package_root(start: Path | None = None) -> Path:
    """Find the local package root from examples or project notebooks."""

    start = (start or Path.cwd()).resolve()
    for base in (start, *start.parents):
        for candidate in (base, base / "package-xyt-gps"):
            if (candidate / "src" / "xyt_gps").exists():
                return candidate.resolve()
    raise FileNotFoundError("Impossible de trouver package-xyt-gps/src/xyt_gps depuis le répertoire courant.")


package_root = find_package_root()
import pandas as pd
import xyt_gps as xyt
pd.set_option("display.max_columns", 80)
print(f"xyt_gps chargé depuis : {xyt.__file__}")
package_root


## 2. Charger les données test

`load_sample_gps()` charge une storyline de démonstration et construit les tables minimales attendues par le package : `storyline`, `trips`, `journeys` et `user_statistics`.

Ici, il n'y a pas d'expérience, pas de phase et pas de chemin vers `Data/`.


In [ ]:
raw = xyt.load_sample_gps(
    user_id="demo_user",
    max_rows=800,
)

raw_tables = {
    "storyline": raw.storyline,
    "trips": raw.trips,
    "journeys": raw.journeys,
    "user_statistics": raw.user_statistics,
}

pd.DataFrame(
    [
        {"table": name, "rows": len(table), "columns": len(table.columns)}
        for name, table in raw_tables.items()
    ]
)


## 3. Valider le schéma brut

Avant de transformer, on vérifie que les tables contiennent les colonnes minimales attendues. Les erreurs bloquantes doivent être corrigées avant de continuer. Les avertissements signalent surtout des enrichissements qui pourraient manquer.


In [ ]:
validation_summary = pd.DataFrame(
    [
        {
            "table": name,
            "ok": report.ok,
            "rows": report.n_rows,
            "columns": report.n_columns,
            "issues": len(report.issues),
        }
        for name, report in raw.validation.items()
    ]
)

validation_summary


In [ ]:
validation_issues = pd.concat(
    [report.to_frame() for report in raw.validation.values()],
    ignore_index=True,
)

validation_issues


## 4. Définir une configuration minimale

Le cas par défaut est volontairement sobre. `ProjectConfig()` ne définit ni `experiment_name`, ni `motiontag_project_name`, ni `period`, ni `phases`.

Cela convient lorsque les tables sont déjà chargées en mémoire et que l'on veut analyser des traces GPS sans protocole expérimental.


In [ ]:
config = xyt.ProjectConfig()
config


## 5. Transformer les traces GPS

`prepare_mobility_dataset()` orchestre la préparation des données : parsing des dates, parsing des géométries, harmonisation des modes et motifs, séparation entre `staypoints` et `legs`, relations avec `trips` et `journeys`, statistiques utilisateur et qualité GPS.

Les paramètres ci-dessous restent visibles pour éviter une transformation opaque.


In [ ]:
dataset = xyt.prepare_mobility_dataset(
    raw,
    config,
    resample_missing_days=False,
    clean_leg_geometries=True,
    add_length_outlier_flags=True,
    add_signal_quality_flags=True,
)

workflow_tables = xyt.mobility_dataset_tables(dataset)

pd.DataFrame(
    [
        {"table": name, "rows": len(table), "columns": len(table.columns)}
        for name, table in workflow_tables.items()
    ]
)


## 6. Vérifier que le cas est bien sans phase

Les tables de mobilité ne reçoivent pas de colonne `phase` si aucune phase n'est définie. Ce n'est pas une erreur : le découpage en phases est une option d'analyse, pas une condition d'entrée.


In [ ]:
pd.DataFrame(
    [
        {"table": name, "has_phase_column": "phase" in table.columns}
        for name, table in workflow_tables.items()
    ]
)


## 7. Inspecter les tables principales

Les `legs` sont les déplacements élémentaires. Les `staypoints` sont les arrêts ou activités. Les `trips` et `journeys` reprennent les niveaux fournis ou dérivés à partir du format MotionTag.


In [ ]:
legs_columns = [
    "leg_id",
    "user_id",
    "started_at",
    "finished_at",
    "mode",
    "mode_niv1",
    "length",
    "duration",
    "rel_max_signalloss",
    "low_quality_legs_1",
]

available_leg_columns = [column for column in legs_columns if column in dataset.legs.columns]
dataset.legs[available_leg_columns].head()


In [ ]:
staypoint_columns = [
    "activity_id",
    "user_id",
    "started_at",
    "finished_at",
    "purpose",
    "purpose_niv1",
    "lon",
    "lat",
]

available_staypoint_columns = [column for column in staypoint_columns if column in dataset.staypoints.columns]
dataset.staypoints[available_staypoint_columns].head()


## 8. Contrôler la qualité du suivi

Le package sépare la transformation des données et la décision de filtrage. On produit donc une table de sélection explicite avant d'exclure éventuellement des utilisateurs.


In [ ]:
quality_report = xyt.build_tracking_quality_report(dataset.user_stats)
quality_report


In [ ]:
selection_table = xyt.build_user_selection_table(
    dataset.user_stats,
    require_tracking_quality=True,
    exclude_bad_signal_users=True,
    max_low_quality_legs_share=0.25,
)

selection_columns = [
    "user_id",
    "tracking_filter_ok",
    "bad_signal_user_filter_ok",
    "low_quality_legs_share_filter_ok",
    "analysis_user_ok",
    "analysis_user_reason",
]

selection_table[[column for column in selection_columns if column in selection_table.columns]]


## 9. Visualiser la participation hebdomadaire

La grille de participation reprend le principe d'une heatmap type GitHub : une ligne par utilisateur, une cellule par semaine, et une intensité verte selon le nombre de jours actifs observés.

Dans cette démo, le sample est pseudonymisé en un seul utilisateur. Sur un jeu multi-utilisateurs, chaque utilisateur aura sa propre ligne.

In [ ]:
participation_grid = xyt.build_weekly_participation_grid(
    dataset.storyline,
    config,
    user_ids=dataset.user_stats["user_id"],
)

participation_summary = xyt.summarize_participation_grid(participation_grid)
participation_summary

In [ ]:
xyt.plot_participation_heatmap(
    participation_grid,
    title="Participation hebdomadaire du sample",
)

## 10. Construire les mobility motifs

Les mobility motifs résument la forme quotidienne des lieux visités. Deux journées peuvent donc avoir le même motif même si les lieux réels ne sont pas les mêmes : par exemple domicile, travail, domicile.

Si une colonne `location_id` existe, elle peut être passée à la fonction. Dans cette démo, le package dérive un identifiant de lieu à partir du motif d'activité et des coordonnées arrondies.

In [ ]:
motifs = xyt.build_mobility_motifs(dataset)

motifs[[
    "user_id_day",
    "n_visits",
    "n_locations",
    "n_transitions",
    "is_round_trip",
    "motif_id",
]].head()


In [ ]:
xyt.summarize_mobility_motifs(motifs).head()


In [ ]:
motif_sequences = xyt.build_mobility_motif_sequences(
    motifs,
    n_days=14,
    align_to_week=False,
)

motif_sequences.head()


## 11. Calculer les indicateurs

`compute_mobility_indicators()` produit trois tables :

- `person_day` : indicateurs par personne, jour et mode ;
- `person_phase` : moyenne par personne et période analytique ;
- `population` : moyenne populationnelle par période analytique et mode.

Comme il n'y a pas de phase dans cet exemple, la période analytique est automatiquement nommée `All`.


In [ ]:
indicators = xyt.compute_mobility_indicators(
    dataset,
    mode_col="mode_niv1",
    include_zero_days=True,
)

pd.DataFrame(
    [
        {"table": name, "rows": len(table), "columns": len(table.columns)}
        for name, table in indicators.tables().items()
    ]
)


In [ ]:
indicators.person_day.head()


In [ ]:
xyt.population_indicator_summary(indicators)


Les mêmes indicateurs peuvent être affichés sous forme de bar charts. Cela permet de contrôler rapidement les ordres de grandeur par mode avant d'exporter les tables.

In [ ]:
xyt.plot_indicator_bars(
    indicators,
    title="Indicateurs population par mode",
)


## 12. Exporter si nécessaire

L'export est désactivé par défaut dans cette démo. L'activer permet d'écrire les états intermédiaires et les rapports qualité pour contrôle externe.


In [ ]:
EXPORT_DEMO = False
export_dir = package_root / "examples" / "outputs" / "demo-gps-generic-no-phase"

if EXPORT_DEMO:
    manifest = xyt.write_mobility_dataset(
        dataset,
        export_dir,
        formats=("csv", "geojson", "parquet"),
        selection_table=selection_table,
    )
    manifest


## 13. Visualiser rapidement les traces

La carte Folium est optionnelle. Elle est utile pour un contrôle visuel, mais elle peut ralentir le notebook si beaucoup de traces sont affichées.


In [ ]:
MAKE_MAP = False

if MAKE_MAP:
    gps_map = xyt.plot_gps_traces(
        dataset,
        sample_n=200,
        random_state=42,
        color_by="mode_niv1",
    )
    gps_map

## À retenir

Pour un usage générique, le package peut fonctionner sans expérimentation et sans phase :

```python
config = xyt.ProjectConfig()
dataset = xyt.prepare_mobility_dataset(raw, config)
indicators = xyt.compute_mobility_indicators(dataset)
```

Le découpage en phases doit être ajouté seulement lorsque la question d'analyse l'exige.
